# 🔭 Step-Back Prompting

**Step-back prompting** (Zheng et al., [arXiv:2310.06117](https://arxiv.org/abs/2310.06117)) improves
retrieval by asking a *more general* question before the specific one.

Faced with *"Did Leonardo da Vinci invent the printing press?"*, a retriever searches for exactly
that — and mostly finds pages about Gutenberg. The knowledge that actually settles the question is
broader: **what did Leonardo actually invent?** Step-back prompting generates that wider question,
retrieves for both, and answers using the combined context.

| | Question used for retrieval | What it surfaces |
|---|---|---|
| **Original** | *"Did Leonardo da Vinci invent the printing press?"* | Printing-press history — Gutenberg |
| **Step-back** | *"What inventions is Leonardo da Vinci associated with?"* | Leonardo's actual body of work |

The second is what lets the model answer *"no, and here is what he did invent instead."*

## Learning Objectives
1. **The specificity trap** — why a narrow question retrieves narrow, often off-target context
2. **Few-shot abstraction** — teaching an LLM to generalize a question by showing it examples
3. **Dual retrieval** — searching with the original *and* the step-back question
4. **Context fusion** — how the `stepback-answer` prompt consumes both result sets
5. **When it helps and when it wastes a call** — the honest tradeoff

## Prerequisites
- A `.env` at the repo root with `EXPERIENTIALLABS_API_KEY` (and `LANGSMITH_API_KEY` for tracing)
- The `ddgs` package for DuckDuckGo search: `uv pip install ddgs`
- Internet access — this notebook retrieves from the live web, not a local vector store

---
## 🧠 Part 1: Why Step Back?

Most RAG failures are retrieval failures. If the right passage never enters the context window, no
amount of prompting recovers it.

Highly specific questions are especially prone to this, because they are often phrased around a
**false premise**. *"Did Leonardo da Vinci invent the printing press?"* embeds an assumption that is
wrong. Searching that phrasing retrieves documents about the printing press — Gutenberg, movable
type, 1440 — none of which discuss Leonardo's inventions.

Stepping back to *"What inventions is Leonardo da Vinci associated with?"* retrieves the aerial
screw, the parachute, the diving suit. That context lets the model answer the original question
correctly and explain *why*.

### Key Concepts:
- **Step-back question**: a deliberately broader reformulation, easier to retrieve good context for.
- **Few-shot abstraction**: the LLM learns the "generalize this" move from worked examples, not rules.
- **Dual context**: both result sets are passed to the final prompt — the step-back context supplies
  background, the original context supplies specifics.

> **Key Insight**: step-back prompting does not make the model smarter about the answer. It changes
> *what gets retrieved*, so the model has the material it needs to reason with.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

from dotenv import load_dotenv
from langsmith import Client

# LangChain core
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.runnables import RunnableLambda

# Web search backend (requires the `ddgs` package)
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

`get_experientiallabs_llm()` reads `EXPERIENTIALLABS_API_KEY` directly from the environment, so
`load_dotenv()` must run **before** any model is created.

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK checks the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already present in `.env` would silently
> win and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Step-Back_Prompting"

print(f"✅ Experiential Labs key: {bool(os.getenv('EXPERIENTIALLABS_API_KEY'))}")
print(f"✅ LangSmith tracing:     {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:     {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the LLM

One model does both jobs here: generating the step-back question, and writing the final answer.

In [ ]:
# ============================================================================
# MODEL INITIALIZATION
# ============================================================================
llm = get_experientiallabs_llm()

print(f"🤖 LLM: {llm.model_name}")

---
## 🎓 Part 3: Teaching Abstraction by Example

"Make this question more general" is hard to specify in prose — too vague and the model rewrites the
question into something unrelated; too strict and it barely changes anything.

Few-shot examples communicate the move far more precisely than instructions do. Notice the pattern
in both: a **yes/no question about one specific thing** becomes an **open question about the
category** it belongs to.

| Input | Output |
|---|---|
| *"Did the Beatles ever write a book?"* | *"What types of creative works did the Beatles produce?"* |
| *"Was Albert Einstein a musician?"* | *"What fields did Albert Einstein work in?"* |

In [ ]:
# ============================================================================
# FEW-SHOT EXAMPLES: Demonstrate the "step back" transformation
# ============================================================================
examples = [
    {
        "input": "Did the Beatles ever write a book?",
        "output": "What types of creative works did the Beatles produce?",
    },
    {
        "input": "Was Albert Einstein a musician?",
        "output": "What fields did Albert Einstein work in?",
    },
]

# Each example becomes a human/ai message pair the model can pattern-match on.
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

print(f"✅ Built few-shot prompt from {len(examples)} examples")

---
## 🔨 Part 4: Build the Step-Back Question Generator

The system message states the task; `few_shot_prompt` expands into the example message pairs; the
user message carries the real question. `StrOutputParser()` reduces the response to a plain string.

In [ ]:
# ============================================================================
# STEP-BACK CHAIN: question -> more general question
# ============================================================================
system_message = (
    "You are an expert at world knowledge. Your task is to step back and "
    "paraphrase a question to a more generic step-back question, which is "
    "easier to answer. Here are a few examples:"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        few_shot_prompt,        # expands into the human/ai example pairs
        ("user", "{question}"),
    ]
)

question_gen = prompt | llm | StrOutputParser()

print("✅ Step-back question generator ready")

### 4.1 Inspect the Assembled Prompt

Worth doing once. `few_shot_prompt` is not a string — it expands into real messages, and seeing the
final message list makes the mechanism concrete rather than magical.

In [ ]:
# ============================================================================
# INTROSPECTION: What the model actually receives
# ============================================================================
question = "Did Leonardo da Vinci invent the printing press?"

for message in prompt.format_messages(question=question):
    role = message.__class__.__name__.replace("Message", "").lower()
    text = message.content if len(message.content) < 110 else message.content[:107] + "..."
    print(f"📋 {role:<7} | {text}")

---
## 🔍 Part 5: See the Transformation

The payoff cell. Compare the question you asked with the question that will actually be searched.

In [ ]:
# ============================================================================
# STEP-BACK IN ACTION: original question vs generated step-back question
# ============================================================================
step_back_question = question_gen.invoke({"question": question})

print(f"❓ Original  : {question}")
print(f"🔭 Step-back : {step_back_question}")

Note what changed. The original presupposes something false (that da Vinci may have invented the
printing press). The step-back question drops that premise and asks about his inventions in general
— which is a question the web can actually answer well.

---
## 🌐 Part 6: Retrieval — Baseline vs Step-Back

Here retrieval is a live DuckDuckGo search rather than a vector store, which makes the difference in
results easy to see. We search with **both** questions and compare what comes back.

> **⚠️ Note**: DuckDuckGo rate-limits aggressively and occasionally fails outright. The wrapper below
> degrades legibly instead of killing the notebook — if you see the warning, wait a minute and re-run.

In [ ]:
# ============================================================================
# RETRIEVER: Live web search, with a legible failure mode
# ============================================================================
search = DuckDuckGoSearchAPIWrapper(max_results=4)


def retriever(query: str) -> str:
    """Run a web search, degrading to a placeholder instead of raising.

    DuckDuckGo rate-limits unpredictably; a hard failure here would break the
    whole chain in Part 7 for a reason unrelated to step-back prompting.
    """
    try:
        return search.run(query)
    except Exception as exc:
        print(f"⚠️  Web search failed for {query!r}: {type(exc).__name__}: {exc}")
        return "(web search unavailable)"


print("✅ Retriever ready")

### 6.1 Baseline — Retrieving With the Original Question

In [ ]:
# ============================================================================
# BASELINE RETRIEVAL: the narrow, premise-laden question
# ============================================================================
normal_context = retriever(question)

print(f"🔍 Results for: {question}\n")
print(normal_context[:700], "...")

### 6.2 Step-Back — Retrieving With the General Question

In [ ]:
# ============================================================================
# STEP-BACK RETRIEVAL: the broader question
# ============================================================================
step_back_context = retriever(step_back_question)

print(f"🔭 Results for: {step_back_question}\n")
print(step_back_context[:700], "...")

### 6.3 Compare What Each Retrieved

A crude but revealing check: how often does each result set actually mention Leonardo, versus
Gutenberg and the printing press?

In [ ]:
# ============================================================================
# COMPARISON: Which context is actually about the right subject?
# ============================================================================
def mentions(text, *terms):
    lowered = text.lower()
    return {term: lowered.count(term.lower()) for term in terms}


print("📊 Term counts in retrieved context\n")
print(f"{'':<14}{'leonardo':>10}{'gutenberg':>12}{'invention':>12}")
for label, ctx in [("baseline", normal_context), ("step-back", step_back_context)]:
    c = mentions(ctx, "leonardo", "gutenberg", "invention")
    print(f"{label:<14}{c['leonardo']:>10}{c['gutenberg']:>12}{c['invention']:>12}")

print("\n💡 The baseline tends to be dominated by printing-press history;")
print("   the step-back context is about da Vinci's actual inventions.")

---
## 🔗 Part 7: The Full RAG Chain

Now assemble everything. The chain builds a dict with three keys, all computed in parallel:

| Key | How it is produced |
|---|---|
| `normal_context` | original question → retriever |
| `step_back_context` | original question → `question_gen` → retriever |
| `question` | the original question, passed straight through |

That dict feeds `langchain-ai/stepback-answer`, a community prompt written to consume exactly these
three fields.

> **Note on `dangerously_pull_public_prompt=True`**: LangSmith now refuses to pull public prompts by
> default, because a prompt manifest can carry serialized objects that execute on deserialization.
> The flag is an explicit acknowledgement. The deprecated `hub.pull()` helper cannot pull public
> prompts at all — it never forwards this flag — so the LangSmith SDK is used directly.

In [ ]:
# ============================================================================
# PROMPT: Pull the community step-back answering prompt
# ============================================================================
response_prompt = Client().pull_prompt(
    "langchain-ai/stepback-answer",
    dangerously_pull_public_prompt=True,   # see the note above
)

print("✅ Pulled 'langchain-ai/stepback-answer'")
print(f"📋 Expects variables: {response_prompt.input_variables}")

In [ ]:
# ============================================================================
# FULL CHAIN: dual retrieval -> combined prompt -> answer
# ============================================================================
chain = (
    {
        # Retrieve with the question as asked.
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve with the generalized question.
        "step_back_context": question_gen | retriever,
        # Keep the original question for the final prompt.
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

print("✅ Step-back RAG chain assembled")

In [ ]:
# ============================================================================
# RUN: Answer the original question using both contexts
# ============================================================================
answer = chain.invoke({"question": question})

print(f"❓ {question}\n")
print(answer)

---
## ⚖️ Part 8: The Tradeoff

Step-back prompting is not free, and it is not universally better.

| | Plain RAG | Step-back RAG |
|---|---|---|
| **LLM calls per query** | 1 (answer) | 2 (step-back question + answer) |
| **Retrieval calls** | 1 | 2 |
| **False-premise questions** | Often confidently wrong | Strong — retrieves the context that corrects the premise |
| **Multi-hop / reasoning questions** | Weak | Helps — background context supports the reasoning |
| **Simple lookups** | Ideal | Wasteful; the step-back question can dilute a precise query |
| **Failure mode** | Misses context | May over-generalize and retrieve something too vague |

**Use step-back when** questions are specific, comparative, or carry assumptions worth checking —
and when your corpus contains the broader background material to retrieve.

**Skip it when** queries are direct lookups (*"what is the refund window?"*), latency matters, or
the corpus is narrow enough that generalizing returns the same documents anyway.

---
## 📝 Summary

### 1. The Problem
- RAG fails when retrieval fails. Narrowly-phrased questions — especially those built on a false
  premise — retrieve documents about the premise rather than about the answer.

### 2. The Technique
- Generate a **more general** question, retrieve for both it and the original, and answer from the
  combined context.
- Part 5 showed the transformation; Part 6 showed the retrieved context genuinely differing.

### 3. Few-Shot Beats Instructions
- "Be more general" is hard to specify in prose. Two worked examples conveyed the pattern —
  specific yes/no question → open question about the category.
- `FewShotChatMessagePromptTemplate` expands into real message pairs (see Part 4.1), not a string.

### 4. Dual Context
- `normal_context` supplies specifics, `step_back_context` supplies background. The
  `langchain-ai/stepback-answer` prompt is built to consume both plus the original question.

### 5. Practical Notes
- Public prompts need `dangerously_pull_public_prompt=True`, and the deprecated `hub.pull()` cannot
  fetch them at all — use the LangSmith `Client` directly.
- Live web search is rate-limited; the retriever degrades to a placeholder so one flaky search does
  not obscure the lesson.

### 6. Cost
- Two LLM calls and two retrievals per question. Worth it for premise-laden or comparative
  questions; wasteful for direct lookups.

### Next Steps
- Inspect these runs in LangSmith under the **Step-Back_Prompting** project — each trace shows the
  step-back generation preceding the answer.
- Compare with the sibling techniques here: `a. Multi_Query` (many rephrasings), `b. RAG_Fusion`
  (rank fusion), `d. HyDE` (embed a hypothetical answer). All four rewrite the query — step-back is
  the one that deliberately makes it *broader*.